# Creating Shells from Disjoint Faces

This notebook demonstrates how to create shells from disjoint (non-touching) faces,
similar to creating a floor plan with rooms that have gaps between them.

**Adapted from topologicpy ShellByDisjointFaces example.**

In topologicpy, `Shell.ByDisjointFaces()` creates a connected shell from faces
that don't share edges, filling the gaps between them.

Since topologic_fast may not have this exact function, we demonstrate:
1. Creating rectangular room faces with gaps
2. Computing connecting faces to fill gaps
3. Building a complete shell
4. Creating a connectivity graph

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import math

print("Libraries imported successfully.")

## 1. Create Room Faces with Gaps

We create a floor plan with multiple rooms. Each room is a rectangular face,
and there are gaps (representing walls) between them.

In [ ]:
def create_rectangle_face(x, y, width, length):
    """
    Create a rectangular face at position (x, y) with given width and length.
    The rectangle is created with its lower-left corner at (x, y).
    """
    # Create vertices
    v0 = tf.Vertex.ByCoordinates(x, y, 0)
    v1 = tf.Vertex.ByCoordinates(x + width, y, 0)
    v2 = tf.Vertex.ByCoordinates(x + width, y + length, 0)
    v3 = tf.Vertex.ByCoordinates(x, y + length, 0)
    
    # Create edges using ByStartVertexEndVertex
    e0 = tf.Edge.ByStartVertexEndVertex(v0, v1)
    e1 = tf.Edge.ByStartVertexEndVertex(v1, v2)
    e2 = tf.Edge.ByStartVertexEndVertex(v2, v3)
    e3 = tf.Edge.ByStartVertexEndVertex(v3, v0)
    
    # Create wire and face
    wire = tf.Wire.ByEdges([e0, e1, e2, e3])
    face = tf.Face.ByWire(wire)
    
    return face

# Define room parameters: (x, y, width, length, name, room_id)
# Gap of 0.1 between rooms (representing wall thickness)
offset = 0.1

room_definitions = [
    # Row 1 (bottom)
    (0 + offset, 0 + offset, 3 - 2*offset, 3 - 2*offset, "Room 1", 1),
    (3 + offset, 0 + offset, 3 - 2*offset, 3 - 2*offset, "Room 2", 2),
    (6 + offset, 0 + offset, 6 - 2*offset, 2.4 - 2*offset, "Room 3", 3),
    
    # Row 2 (middle-bottom)
    (0 + offset, 3 + offset, 5 - 2*offset, 3 - 2*offset, "Room 4", 4),
    (6 + offset, 3 + offset, 6 - 2*offset, 3 - 2*offset, "Room 5", 5),
    
    # Row 3 (top)
    (0 + offset, 6 + offset, 6 - 2*offset, 3 - 2*offset, "Room 6", 6),
    (6 + offset, 6 + offset, 6 - 2*offset, 3 - 2*offset, "Room 7", 7),
]

# Create room faces
rooms = []
room_names = []
room_ids = []

for x, y, w, l, name, room_id in room_definitions:
    face = create_rectangle_face(x, y, w, l)
    rooms.append(face)
    room_names.append(name)
    room_ids.append(room_id)

print(f"Created {len(rooms)} room faces:")
for i, (name, room_id) in enumerate(zip(room_names, room_ids)):
    area = rooms[i].Area()
    print(f"  {name} (ID: {room_id}): {area:.2f} sq units")

## 2. Create External Boundary

We create an external boundary that encompasses all rooms.
This will have an inner boundary (inset) and outer boundary.

In [ ]:
# Create outer boundary
outer_boundary = create_rectangle_face(0, 0, 12, 9)

# Create inner boundary (slightly inset)
inner_boundary = create_rectangle_face(offset, offset, 12 - 2*offset, 9 - 2*offset)

# NOTE: In topologicpy, you would create a face with a hole:
#   rect = Face.ByWires(rect2, [rect1])
# This represents the wall area around all rooms.

print(f"Outer boundary area: {outer_boundary.Area():.2f} sq units")
print(f"Inner boundary area: {inner_boundary.Area():.2f} sq units")
print(f"Total room area: {sum(r.Area() for r in rooms):.2f} sq units")

## 3. Visualize Rooms and Boundary

In [ ]:
# Color scale for rooms (thermal-like)
room_colors = [
    '#440154',  # Room 1 - dark purple
    '#3B528B',  # Room 2 - blue
    '#21918C',  # Room 3 - teal
    '#5EC962',  # Room 4 - green
    '#FDE725',  # Room 5 - yellow
    '#F89441',  # Room 6 - orange
    '#D62728',  # Room 7 - red
]

def visualize_floor_plan(rooms, room_names, room_colors, boundary=None, title="Floor Plan"):
    """Visualize the floor plan in 2D."""
    fig = go.Figure()
    
    # Draw boundary first (if provided)
    if boundary is not None:
        vertices = boundary.Vertices()
        coords = [v.Coordinates() for v in vertices]
        x = [c[0] for c in coords] + [coords[0][0]]
        y = [c[1] for c in coords] + [coords[0][1]]
        
        fig.add_trace(go.Scatter(
            x=x, y=y,
            fill='toself',
            fillcolor='lightgray',
            line=dict(color='black', width=2),
            name='Boundary',
            hoverinfo='name'
        ))
    
    # Draw rooms
    for room, name, color in zip(rooms, room_names, room_colors):
        vertices = room.Vertices()
        coords = [v.Coordinates() for v in vertices]
        x = [c[0] for c in coords] + [coords[0][0]]
        y = [c[1] for c in coords] + [coords[0][1]]
        
        fig.add_trace(go.Scatter(
            x=x, y=y,
            fill='toself',
            fillcolor=color,
            line=dict(color='black', width=1),
            name=name,
            hoverinfo='name'
        ))
        
        # Add room label at centroid
        cx = sum(c[0] for c in coords[:4]) / 4
        cy = sum(c[1] for c in coords[:4]) / 4
        
        fig.add_annotation(
            x=cx, y=cy,
            text=name,
            showarrow=False,
            font=dict(size=10, color='white')
        )
    
    fig.update_layout(
        title=title,
        xaxis=dict(
            title='X',
            scaleanchor='y',
            scaleratio=1,
            range=[-0.5, 12.5]
        ),
        yaxis=dict(
            title='Y',
            range=[-0.5, 9.5]
        ),
        width=800,
        height=600,
        showlegend=True
    )
    
    return fig

fig_plan = visualize_floor_plan(rooms, room_names, room_colors, outer_boundary, "Disjoint Room Faces")
fig_plan.show()

## 4. Create Shell from Disjoint Faces

In topologicpy, `Shell.ByDisjointFaces()` would:
1. Take the external boundary and individual room faces
2. Fill gaps between rooms with connecting faces
3. Return a complete shell

Since topologic_fast may not have this exact function, we create a simple shell
from all the room faces.

**NOTE:** `Shell.ByDisjointFaces` is a complex operation that may not be available in topologic_fast.
We demonstrate a simplified approach.

In [ ]:
# NOTE: In topologicpy, you would use:
#   shell = Shell.ByDisjointFaces(
#       externalBoundary=rect,
#       faces=rooms,
#       maximumGap=0.8,
#       mergeJunctions=True,
#       threshold=0.8,
#       transferDictionaries=True,
#       tolerance=0.0001
#   )
#
# For topologic_fast, we create a simple shell from the room faces:

try:
    shell = tf.Shell.ByFaces(rooms)
    print(f"Created Shell:")
    print(f"  Number of faces: {shell.NumFaces()}")
    print(f"  Total area: {shell.Area():.2f} sq units")
except Exception as e:
    print(f"Note: Shell.ByFaces may not work for disjoint faces.")
    print(f"Error: {e}")
    print("\nUsing individual faces for visualization instead.")
    shell = None

## 5. Compute Room Adjacency

Even though rooms don't share edges directly (due to gaps),
we can determine which rooms are "adjacent" based on proximity.

Two rooms are considered adjacent if their bounding boxes are close enough
(separated by less than the gap threshold).

In [ ]:
def get_face_bounds(face):
    """Get bounding box of a face."""
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    x_coords = [c[0] for c in coords]
    y_coords = [c[1] for c in coords]
    
    return {
        'min_x': min(x_coords),
        'max_x': max(x_coords),
        'min_y': min(y_coords),
        'max_y': max(y_coords)
    }

def are_rooms_adjacent(face1, face2, gap_threshold=0.3):
    """
    Check if two room faces are adjacent (close enough to be considered neighbors).
    
    Two rooms are adjacent if:
    - They share an edge dimension (overlap in X or Y)
    - The gap between them is less than the threshold
    """
    b1 = get_face_bounds(face1)
    b2 = get_face_bounds(face2)
    
    # Check horizontal adjacency (side by side)
    y_overlap = not (b1['max_y'] < b2['min_y'] or b2['max_y'] < b1['min_y'])
    
    if y_overlap:
        # Check if horizontally adjacent
        gap_x = max(b1['min_x'] - b2['max_x'], b2['min_x'] - b1['max_x'])
        if 0 < gap_x <= gap_threshold:
            return True
    
    # Check vertical adjacency (stacked)
    x_overlap = not (b1['max_x'] < b2['min_x'] or b2['max_x'] < b1['min_x'])
    
    if x_overlap:
        # Check if vertically adjacent
        gap_y = max(b1['min_y'] - b2['max_y'], b2['min_y'] - b1['max_y'])
        if 0 < gap_y <= gap_threshold:
            return True
    
    return False

# Build adjacency list
adjacency = {i: [] for i in range(len(rooms))}

for i in range(len(rooms)):
    for j in range(i + 1, len(rooms)):
        if are_rooms_adjacent(rooms[i], rooms[j], gap_threshold=0.25):
            adjacency[i].append(j)
            adjacency[j].append(i)

print("Room Adjacency:")
print("-" * 40)
for i, neighbors in adjacency.items():
    neighbor_names = [room_names[n] for n in neighbors]
    print(f"  {room_names[i]}: {', '.join(neighbor_names) if neighbor_names else 'No neighbors'}")

## 6. Create and Visualize Connectivity Graph

We create a graph where:
- Each vertex represents a room (at its centroid)
- Each edge represents adjacency between rooms

In [ ]:
def get_face_centroid(face):
    """Compute centroid of a face."""
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    cx = sum(c[0] for c in coords) / len(coords)
    cy = sum(c[1] for c in coords) / len(coords)
    cz = sum(c[2] for c in coords) / len(coords)
    
    return (cx, cy, cz)

# Get centroids
centroids = [get_face_centroid(room) for room in rooms]

# Create graph vertices at centroids
graph_vertices = [tf.Vertex.ByCoordinates(c[0], c[1], c[2]) for c in centroids]

# Create graph edges based on adjacency using ByStartVertexEndVertex
graph_edges = []
for i, neighbors in adjacency.items():
    for j in neighbors:
        if i < j:  # Avoid duplicates
            edge = tf.Edge.ByStartVertexEndVertex(graph_vertices[i], graph_vertices[j])
            graph_edges.append(edge)

print(f"Graph vertices: {len(graph_vertices)}")
print(f"Graph edges: {len(graph_edges)}")

In [ ]:
def visualize_floor_plan_with_graph(rooms, room_names, room_colors, centroids, adjacency, title="Floor Plan with Graph"):
    """Visualize floor plan with connectivity graph overlay."""
    fig = go.Figure()
    
    # Draw rooms
    for room, name, color in zip(rooms, room_names, room_colors):
        vertices = room.Vertices()
        coords = [v.Coordinates() for v in vertices]
        x = [c[0] for c in coords] + [coords[0][0]]
        y = [c[1] for c in coords] + [coords[0][1]]
        
        fig.add_trace(go.Scatter(
            x=x, y=y,
            fill='toself',
            fillcolor=color,
            line=dict(color='black', width=1),
            name=name,
            hoverinfo='name',
            opacity=0.7
        ))
    
    # Draw graph edges
    for i, neighbors in adjacency.items():
        for j in neighbors:
            if i < j:
                p1 = centroids[i]
                p2 = centroids[j]
                
                fig.add_trace(go.Scatter(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    mode='lines',
                    line=dict(color='red', width=3),
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    # Draw graph vertices (centroids)
    cx = [c[0] for c in centroids]
    cy = [c[1] for c in centroids]
    
    fig.add_trace(go.Scatter(
        x=cx, y=cy,
        mode='markers+text',
        marker=dict(size=25, color='white', line=dict(color='red', width=3)),
        text=[str(room_ids[i]) for i in range(len(centroids))],
        textposition='middle center',
        textfont=dict(size=12, color='red'),
        hovertext=room_names,
        hoverinfo='text',
        name='Room Centroids'
    ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(
            title='X',
            scaleanchor='y',
            scaleratio=1,
            range=[-0.5, 12.5]
        ),
        yaxis=dict(
            title='Y',
            range=[-0.5, 9.5]
        ),
        width=800,
        height=600,
        showlegend=True
    )
    
    return fig

fig_graph = visualize_floor_plan_with_graph(rooms, room_names, room_colors, centroids, adjacency)
fig_graph.show()

## 7. 3D Visualization

We can also visualize the floor plan in 3D, giving each room some height.

In [ ]:
def visualize_floor_plan_3d(rooms, room_names, room_colors, height=0.01):
    """Create 3D visualization of floor plan."""
    fig = go.Figure()
    
    for room, name, color in zip(rooms, room_names, room_colors):
        vertices = room.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        x = [c[0] for c in coords]
        y = [c[1] for c in coords]
        z = [height for _ in coords]  # Slight elevation
        
        # Draw face
        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            color=color,
            opacity=1.0,
            alphahull=0,
            name=name,
            showlegend=True,
            hoverinfo='name'
        ))
        
        # Draw edges
        for k in range(len(coords)):
            p1 = coords[k]
            p2 = coords[(k + 1) % len(coords)]
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[height, height],
                mode='lines',
                line=dict(color='black', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    fig.update_layout(
        title='3D Floor Plan',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=0, y=-1.5, z=1.5))
        ),
        width=800,
        height=600
    )
    
    return fig

fig_3d = visualize_floor_plan_3d(rooms, room_names, room_colors)
fig_3d.show()

## 8. Room Statistics

In [ ]:
# Calculate statistics
total_area = sum(room.Area() for room in rooms)
building_area = outer_boundary.Area()
circulation_area = building_area - total_area

print("Floor Plan Statistics")
print("=" * 50)
print(f"\nTotal building area: {building_area:.2f} sq units")
print(f"Total room area: {total_area:.2f} sq units")
print(f"Circulation/wall area: {circulation_area:.2f} sq units")
print(f"Efficiency ratio: {(total_area / building_area) * 100:.1f}%")

print("\nRoom Details:")
print("-" * 50)
print(f"{'Room':<15} {'Area':>10} {'Connections':>12}")
print("-" * 50)

for i, (room, name) in enumerate(zip(rooms, room_names)):
    area = room.Area()
    connections = len(adjacency[i])
    print(f"{name:<15} {area:>10.2f} {connections:>12}")

## Summary

This notebook demonstrated how to work with disjoint faces to create floor plans using topologic_fast.

### Key Operations:

1. **Face Creation**: Built rectangular room faces using `tf.Vertex.ByCoordinates()`, `tf.Edge.ByStartVertexEndVertex()`, `tf.Wire.ByEdges()`, and `tf.Face.ByWire()`
2. **Adjacency Detection**: Computed which rooms are neighbors based on proximity
3. **Graph Construction**: Created vertices and edges representing room connectivity
4. **Visualization**: 2D and 3D views using Plotly

### Differences from topologicpy:

- **No `Shell.ByDisjointFaces()`**: This function automatically fills gaps between faces. We manually computed adjacency instead.
- **No Dictionary support**: Cannot attach metadata (like room IDs) to topologies
- **No `Graph.ByTopology(shell)`**: Built graph manually from adjacency
- **No `Face.ByWires(outer, [inner])`**: Cannot create faces with holes directly

### Applications:

- Architectural floor plan analysis
- Space syntax and connectivity studies
- Building Information Modeling (BIM)
- Wayfinding and navigation analysis